# 3 · Read a sweep you have already run — free

**No API calls.** This reads cell files off disk and renders the same figures the
session shows. Point `SWEEP_DIR` at any directory the sweep wrote.

If you have not run one, notebook 2 is cheaper than a sweep, or:

```
uv run python demos/04_hill_climbing_loop/sweep.py --profile smoke --foreground
```

> Imports from `loopeng`, no loop logic — and no chart logic either. Every figure here
> is built by `loopeng.sweep.charts`, which is the same code the entry point calls, so
> a figure in this notebook cannot disagree with one on the projector.

In [ ]:
import os
from pathlib import Path

# Jupyter starts the kernel in the notebook's own directory, and
# everything in this repository is addressed from the root: `.env`,
# the warehouse, `gold/`, `results/`. Move there once. Idempotent, so
# re-running the cell is a no-op.
if not Path("pyproject.toml").exists():
    os.chdir("..")
print("working from", Path.cwd().name)


In [ ]:
from pathlib import Path

SWEEP_DIR = Path("results/sweep")

## The pre-registration — printed before the first cell of a run

Stated up front so a hypothesis cannot be chosen after the numbers land. It names what
the design can detect, what it cannot, and what it already knows it cannot.

In [ ]:
from loopeng.sweep.orchestrator import load_all, pre_registration

cells = load_all(SWEEP_DIR)
print(f"{len(cells)} cell(s) on disk, "
      f"{sum(1 for c in cells if c['complete'])} complete")

In [ ]:
n_items = max((c["n_requested"] for c in cells), default=0)
print(pre_registration(n_items) if n_items else "no cells yet — run a sweep first")

## What each cell measured

Never blank, never zero, never a guess: a cell with nothing landed says *not yet
measured*, one still running says so, and one the clock or a Ctrl-C ended says it is
final at a smaller `n`.

In [ ]:
for cell in sorted(cells, key=lambda c: c["key"]):
    print(f"{cell['label']:<34} {cell['silent_error_rate']}")

## The figures

Drawn from the cells above, by the same module the entry point uses.

In [ ]:
%matplotlib inline
from loopeng.sweep.charts import cost_chart, dial_chart

dial_chart(cells)

In [ ]:
cost_chart(cells)

## The comparisons, and what may not be said about them

A comparison that could not be tested is listed **with its reason** rather than
omitted, because a shorter list is indistinguishable from a shorter finding. A
cross-model pair gets no p-value at all — that refusal is in code, not in a caption.

In [ ]:
from loopeng.sweep.charts import delta_chart
from loopeng.sweep.render import comparisons_for

comparisons = comparisons_for(cells)
for comparison in comparisons:
    print(f"[{comparison.kind}] {comparison.label_a} -> {comparison.label_b}")
    print(f"    {comparison.reading()}")
    print(f"    {comparison.provenance()}\n")

In [ ]:
delta_chart(comparisons)

## Coverage against precision — the abstention curve

Declining is a choice, and this is the trade it buys. The cell it is drawn from is
named rather than guessed: if that cell is absent, `curve_cell` **raises** instead of
substituting the largest one it can find, and this panel says so rather than rendering
*not yet measured* over a directory that is not empty.

In [ ]:
from loopeng.sweep.charts import abstention_chart
from loopeng.sweep.render import abstention_panel

points, refusal = abstention_panel(cells)
print(refusal or f"{len(points)} threshold(s) on the curve")
abstention_chart(points, refusal=refusal)

## The terminal summary, unabridged

The same lines the chart entry point prints. Nothing is summarised away.

In [ ]:
from loopeng.sweep.render import summarise

for line in summarise(cells, comparisons, SWEEP_DIR, []):
    print(line)